# 01a — Data Cleaning

**Objective**: Apply all cleaning steps identified in notebook 01 to produce `factory_cleaned.csv`.

**Input**: `data/raw/factory_2023_2025.csv` (4,615 rows after footnote filter)

**Output**: `data/processed/factory_cleaned.csv`

---

## Cleaning Steps
1. Load raw data & filter footnote rows
2. Convert numeric columns (string → float)
3. Parse `Contract Date` → datetime, extract `Year`, `Month`, `Quarter`
4. Parse `Tenure` → `Lease_Duration`, `Remaining_Lease_Years`
5. Drop rows
   - Freehold properties (no lease expiry → `Remaining_Lease_Years` undefined)
   - 999-year leases (effectively freehold — perpetual ownership, pricing is not lease-driven)
   - Land transactions (`Type Of Area = 'Land'`)
   - Floor Level `'-'` → rename to `'Unknown'` (after Land rows removed)
   - Subsale (`Type of Sale = 'Subsale'`)
   - Duplicates (composite key)
   - Null `Unit Price ($ psf)`
6. Drop columns (`Property Type`, `Price`, `Tenure`)
7. Join macro indicators
8. Validate & save

## 1. Imports & Load Raw Data

In [1]:
import re
from datetime import datetime
from pathlib import Path

import pandas as pd

RAW_PATH = "../data/raw/factory_2023_2025.csv"
OUT_PATH = "../data/processed/factory_cleaned.csv"
EXTERNAL = "../data/raw/external"

df = pd.read_csv(RAW_PATH)

# Filter footnote/caveat rows — keep only rows where Contract Date matches DD/MM/YYYY
df = df[df["Contract Date"].str.match(r"^\d{2}/\d{2}/\d{4}$", na=False)].reset_index(drop=True)

print(f"Raw rows (after footnote filter): {len(df)}")
print(f"Columns: {list(df.columns)}")

Raw rows (after footnote filter): 4615
Columns: ['Contract Date', 'Project Name', 'Street Name', 'Property Type', 'Type Of Area', 'Area (sqft)', 'Floor Level', 'Price', 'Unit Price ($ psf)', 'Tenure', 'Type of Sale', 'Region', 'Planning Area', 'Postal District', 'Postal Sector']


## 2. Convert Numeric Columns

`Area (sqft)`, `Price`, and `Unit Price ($ psf)` are stored as strings with comma separators.

In [2]:
for col in ["Area (sqft)", "Price", "Unit Price ($ psf)"]:
    df[col] = df[col].str.replace(",", "", regex=False).astype(float)

print("Converted to float:")
print(df[["Area (sqft)", "Price", "Unit Price ($ psf)"]].dtypes)

Converted to float:
Area (sqft)           float64
Price                 float64
Unit Price ($ psf)    float64
dtype: object


## 3. Parse Contract Date → Year, Month, Quarter

In [3]:
df["Contract Date"] = pd.to_datetime(df["Contract Date"], dayfirst=True)
df["Year"]    = df["Contract Date"].dt.year
df["Month"]   = df["Contract Date"].dt.month
df["Quarter"] = df["Contract Date"].dt.quarter

print("Date range:", df["Contract Date"].min().date(), "to", df["Contract Date"].max().date())
print("Year distribution:")
print(df["Year"].value_counts().sort_index())

Date range: 2023-01-03 to 2025-12-31
Year distribution:
Year
2023    1483
2024    1621
2025    1511
Name: count, dtype: int64


## 4. Parse Tenure → Lease_Duration, Remaining_Lease_Years

Extract lease duration (years) and compute remaining lease as of the contract date.

- Freehold and unparseable tenures return `None` — these rows are dropped in Step 5.
- Edge cases (`30+30 yrs`, `28.08 yrs`, entries with `ETC` suffix) are handled by the regex
  where possible; otherwise return `None` and are dropped.

In [4]:
def parse_tenure(tenure_val, contract_date):
    """Extract (Lease_Duration, Remaining_Lease_Years) from a raw Tenure string.
    Returns (None, None) for Freehold or unparseable formats.
    """
    if pd.isna(tenure_val):
        return None, None
    val = str(tenure_val).strip()
    if "Freehold" in val:
        return None, None
    m = re.match(r"(\d+(?:\.\d+)?)\s*yrs?\s+from\s+(\d{2}/\d{2}/\d{4})", val)
    if not m:
        return None, None
    duration = float(m.group(1))
    start    = datetime.strptime(m.group(2), "%d/%m/%Y")
    expiry   = start.replace(year=start.year + int(duration))
    remaining = round((expiry - contract_date.to_pydatetime()).days / 365.25, 1)
    return duration, remaining


parsed = df.apply(
    lambda row: parse_tenure(row["Tenure"], row["Contract Date"]), axis=1
)
df["Lease_Duration"]         = parsed.apply(lambda x: x[0])
df["Remaining_Lease_Years"]  = parsed.apply(lambda x: x[1])

print(f"Parseable leasehold rows: {df['Lease_Duration'].notna().sum()}")
print(f"Unparseable / Freehold:   {df['Lease_Duration'].isna().sum()}")
print()
print("Lease_Duration distribution:")
print(df["Lease_Duration"].value_counts().sort_index())

Parseable leasehold rows: 3942
Unparseable / Freehold:   673

Lease_Duration distribution:
Lease_Duration
19.00        5
20.00        2
26.00       64
28.08        1
30.00     1139
41.00        7
45.00        2
56.00        1
57.00       54
58.00       25
60.00     2415
63.00       40
99.00      147
999.00      40
Name: count, dtype: int64


## 5. Drop Rows

Order matters:
1. Drop Freehold first (sets up Remaining_Lease_Years = null as the filter)
2. Drop Land transactions
3. Rename Floor Level `'-'` to `'Unknown'` (Land rows already gone, so no Land rows get renamed)
4. Drop Subsale
5. Drop duplicates using composite key
6. Drop null Unit Price

In [5]:
COMPOSITE_KEY = [
    "Contract Date", "Street Name", "Property Type", "Type Of Area",
    "Area (sqft)", "Floor Level", "Unit Price ($ psf)",
    "Type of Sale", "Tenure", "Region", "Planning Area",
]

steps = [
    ("Freehold (Remaining_Lease_Years is null)",   lambda d: d[d["Remaining_Lease_Years"].notna()]),
    ("999-year leases (effectively freehold)",      lambda d: d[d["Lease_Duration"] != 999]),
    ("Land transactions (Type Of Area = 'Land')",  lambda d: d[d["Type Of Area"] != "Land"]),
    ("Subsale",                                    lambda d: d[d["Type of Sale"] != "Subsale"]),
    ("Duplicates (composite key)",                 lambda d: d.drop_duplicates(subset=COMPOSITE_KEY)),
    ("Null Unit Price ($ psf)",                    lambda d: d.dropna(subset=["Unit Price ($ psf)"])),
]

print(f"{'Step':<45} {'Dropped':>8}  {'Remaining':>10}")
print("-" * 68)
print(f"{'Start':<45} {'':>8}  {len(df):>10}")

for label, fn in steps:
    before = len(df)
    df = fn(df).reset_index(drop=True)
    print(f"{label:<45} {before - len(df):>8}  {len(df):>10}")

# Rename Floor Level '-' → 'Unknown' after Land rows are gone
renamed = (df["Floor Level"] == "-").sum()
df["Floor Level"] = df["Floor Level"].replace("-", "Unknown")
print(f"\nFloor Level '-' renamed to 'Unknown': {renamed} rows")
print(f"\nFinal row count: {len(df)}")

Step                                           Dropped   Remaining
--------------------------------------------------------------------
Start                                                         4615
Freehold (Remaining_Lease_Years is null)           673        3942
999-year leases (effectively freehold)              40        3902
Land transactions (Type Of Area = 'Land')           22        3880
Subsale                                              0        3880
Duplicates (composite key)                          91        3789
Null Unit Price ($ psf)                              0        3789

Floor Level '-' renamed to 'Unknown': 90 rows

Final row count: 3789


## 6. Drop Columns

| Column | Reason |
|--------|--------|
| `Property Type` | Constant column — all rows are `Multiple-User Factory` |
| `Price` | Total transaction price — not needed; `Unit Price ($ psf)` is the target |
| `Tenure` | Replaced by `Lease_Duration` and `Remaining_Lease_Years` |
| `Type Of Area` | Constant column after Land rows dropped — all remaining rows are `Strata` |

In [6]:
DROP_COLS = ["Property Type", "Price", "Tenure", "Type Of Area"]
df = df.drop(columns=DROP_COLS)

print(f"Dropped columns: {DROP_COLS}")
print(f"Remaining columns ({len(df.columns)}): {list(df.columns)}")

Dropped columns: ['Property Type', 'Price', 'Tenure', 'Type Of Area']
Remaining columns (16): ['Contract Date', 'Project Name', 'Street Name', 'Area (sqft)', 'Floor Level', 'Unit Price ($ psf)', 'Type of Sale', 'Region', 'Planning Area', 'Postal District', 'Postal Sector', 'Year', 'Month', 'Quarter', 'Lease_Duration', 'Remaining_Lease_Years']


## 7. Join Macro Indicators

Macro data is available at quarterly or monthly granularity. Join keys:

| Source file | Granularity | Join key |
|-------------|-------------|----------|
| `gdp_quarterly.csv` | Quarterly | `Year + Quarter` |
| `cpi_quarterly.csv` | Quarterly | `Year + Quarter` |
| `unemployment_quarterly.csv` | Quarterly | `Year + Quarter` |
| `industrial_property_price_index.csv` | Quarterly | `Year + Quarter` |
| `interest_rates_monthly.csv` | Monthly | `Year + Month` |
| `construction_material_prices_monthly.csv` | Monthly | `Year + Month` |

In [7]:
def parse_quarterly(path, period_col="Period"):
    """Parse 'YYYY #Q' period strings into Year + Quarter columns."""
    m = pd.read_csv(path)
    m = m.rename(columns={period_col: "_period"})
    m["_period"] = m["_period"].str.replace(" ", "")  # normalise '2023 1Q' → '20231Q'
    m["Year"]    = m["_period"].str[:4].astype(int)
    m["Quarter"] = m["_period"].str.extract(r"(\d)Q").iloc[:, 0].astype(int)
    return m.drop(columns=["_period"])


def parse_monthly(path, period_col="Period"):
    """Parse 'YYYY-MM' period strings into Year + Month columns."""
    m = pd.read_csv(path)
    m = m.rename(columns={period_col: "_period"})
    m["Year"]  = m["_period"].str[:4].astype(int)
    m["Month"] = m["_period"].str[5:7].astype(int)
    return m.drop(columns=["_period"])


# Load macro sources
gdp        = parse_quarterly(f"{EXTERNAL}/gdp_quarterly.csv")
cpi        = parse_quarterly(f"{EXTERNAL}/cpi_quarterly.csv")
unemp      = parse_quarterly(f"{EXTERNAL}/unemployment_quarterly.csv")
interest   = parse_monthly(f"{EXTERNAL}/interest_rates_monthly.csv")
materials  = parse_monthly(f"{EXTERNAL}/construction_material_prices_monthly.csv")

# Price Index has format '2023Q1' — parse manually
price_idx  = pd.read_csv(f"{EXTERNAL}/industrial_property_price_index.csv")
price_idx["Year"]    = price_idx["Quarter"].str[:4].astype(int)
price_idx["Quarter"] = price_idx["Quarter"].str[-1].astype(int)

# Join quarterly indicators on Year + Quarter
for macro_df in [gdp, cpi, unemp, price_idx]:
    df = df.merge(macro_df, on=["Year", "Quarter"], how="left")

# Join monthly indicators on Year + Month
for macro_df in [interest, materials]:
    df = df.merge(macro_df, on=["Year", "Month"], how="left")

macro_cols = [
    "GDP_YoY_Growth_Rate", "CPI_All_Items", "Unemployment_Rate", "Price_Index",
    "5Y_Bond_Yield", "2Y_Bond_Yield", "10Y_Bond_Yield", "1Y_TBills",
    "15Y_Bond_Yield", "20Y_Bond_Yield", "SORA_3M_Compounded",
    "Cement_Bulk_Per_Tonne", "Steel_Rebar_Per_Tonne",
    "Granite_20mm_Per_Tonne", "Concreting_Sand_Per_Tonne", "Ready_Mixed_Concrete_Per_m3",
]
print("Macro join — null counts:")
print(df[macro_cols].isnull().sum().to_string())

Macro join — null counts:
GDP_YoY_Growth_Rate            0
CPI_All_Items                  0
Unemployment_Rate              0
Price_Index                    0
5Y_Bond_Yield                  0
2Y_Bond_Yield                  0
10Y_Bond_Yield                 0
1Y_TBills                      0
15Y_Bond_Yield                 0
20Y_Bond_Yield                 0
SORA_3M_Compounded             0
Cement_Bulk_Per_Tonne          0
Steel_Rebar_Per_Tonne          0
Granite_20mm_Per_Tonne         0
Concreting_Sand_Per_Tonne      0
Ready_Mixed_Concrete_Per_m3    0


## 8. Validate & Save

In [8]:
# Contract assertions — dataset must meet these before saving
assert df["Remaining_Lease_Years"].isna().sum() == 0,  "Freehold rows not fully removed"
assert (df["Lease_Duration"] == 999).sum() == 0,        "999-year lease rows not removed"
assert (df["Type of Sale"] == "Subsale").sum() == 0,    "Subsale rows not fully removed"
assert df["Unit Price ($ psf)"].isna().sum() == 0,      "Null Unit Price rows remain"
assert (df["Floor Level"] == "-").sum() == 0,           "Floor Level '-' not renamed"
assert df[macro_cols].isnull().sum().sum() == 0,        "Macro join has missing values"

print("All assertions passed.")
print(f"\nFinal shape: {df.shape}")
print(f"\nColumn list ({len(df.columns)}):")
for col in df.columns:
    print(f"  {col}")

print(f"\nFloor Level values:")
print(df["Floor Level"].value_counts())

print(f"\nType of Sale values:")
print(df["Type of Sale"].value_counts())

print(f"\nLease_Duration distribution:")
print(df["Lease_Duration"].value_counts().sort_index())

All assertions passed.

Final shape: (3789, 32)

Column list (32):
  Contract Date
  Project Name
  Street Name
  Area (sqft)
  Floor Level
  Unit Price ($ psf)
  Type of Sale
  Region
  Planning Area
  Postal District
  Postal Sector
  Year
  Month
  Quarter
  Lease_Duration
  Remaining_Lease_Years
  GDP_YoY_Growth_Rate
  CPI_All_Items
  Unemployment_Rate
  Price_Index
  5Y_Bond_Yield
  2Y_Bond_Yield
  10Y_Bond_Yield
  1Y_TBills
  15Y_Bond_Yield
  20Y_Bond_Yield
  SORA_3M_Compounded
  Cement_Bulk_Per_Tonne
  Steel_Rebar_Per_Tonne
  Granite_20mm_Per_Tonne
  Concreting_Sand_Per_Tonne
  Ready_Mixed_Concrete_Per_m3

Floor Level values:
Floor Level
Non-First Floor    3284
First Floor         415
Unknown              90
Name: count, dtype: int64

Type of Sale values:
Type of Sale
Resale      3723
New Sale      66
Name: count, dtype: int64

Lease_Duration distribution:
Lease_Duration
19.0       5
20.0       2
26.0      64
30.0    1105
41.0       7
45.0       2
57.0      54
58.0      24
60.0 

In [9]:
df.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}")
print(f"Shape: {df.shape}")

Saved: ../data/processed/factory_cleaned.csv


Shape: (3789, 32)
